# Colab accuracy train

Trains Softmax and CORAL on a GPU, scores the locked test set **once**, and downloads the model files.

Do this before the first code cell:

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.**
3. When a cell asks, upload `kaggle.json`, then upload `dr_colab_code.zip` (unless you set a Drive path).
4. Leave this tab open. On a T4 this is usually **1 to 2 hours**. If the session disconnects, the models are lost and this notebook has to be run again.
5. The last cell downloads `colab_models.zip`. Put that zip in the project folder and send it back.

The test split is not used for training. Validation chooses the CORAL threshold. The deployed model is replaced only if test accuracy is higher than 0.627.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__)
print("GPU", gpus)
if not gpus:
    raise SystemExit("No GPU. Use Runtime → Change runtime type → T4 GPU, then run this cell again.")


In [ ]:
%pip install -q opencv-python-headless kaggle


## Project code

Leave `ZIP_ON_DRIVE` empty to upload `dr_colab_code.zip` from your computer.

If the zip is already on Google Drive, mount happens automatically when you set a path such as `/content/drive/MyDrive/dr_colab_code.zip`.


In [ ]:
import shutil
import zipfile
from pathlib import Path

from google.colab import files

ROOT = Path("/content/dr")
ZIP_ON_DRIVE = ""  # example: "/content/drive/MyDrive/dr_colab_code.zip"

if ZIP_ON_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    zip_path = Path(ZIP_ON_DRIVE)
else:
    print("Upload dr_colab_code.zip")
    uploaded = files.upload()
    zip_path = Path("/content") / next(iter(uploaded))

if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(ROOT)

if not (ROOT / "src" / "dr_detect").exists():
    raise SystemExit("Zip layout is wrong. Expected a src/dr_detect folder inside dr_colab_code.zip.")
print("Unpacked", ROOT)


## Kaggle login

Upload the `kaggle.json` from this project’s `.kaggle` folder. It is not inside the code zip.


In [ ]:
import os
from pathlib import Path

from google.colab import files

dest = Path("/root/.kaggle")
dest.mkdir(parents=True, exist_ok=True)
cred = dest / "kaggle.json"
if not cred.exists():
    print("Upload kaggle.json")
    uploaded = files.upload()
    raw = uploaded.get("kaggle.json")
    if raw is None:
        raw = next(iter(uploaded.values()))
    cred.write_bytes(raw)
os.chmod(cred, 0o600)
os.environ["KAGGLE_CONFIG_DIR"] = str(dest)
print("Kaggle credentials ready")


## Download the fundus images

The code zip has the locked split lists only. Images are downloaded here so the upload stays small.


In [ ]:
import subprocess
from pathlib import Path

raw = ROOT / "data" / "raw"
aptos = ROOT / "data" / "external" / "aptos2019"
raw.mkdir(parents=True, exist_ok=True)
aptos.mkdir(parents=True, exist_ok=True)

def image_count(folder: Path) -> int:
    n = 0
    for pattern in ("*.png", "*.jpg", "*.jpeg"):
        n += sum(1 for _ in folder.rglob(pattern))
    return n

if image_count(raw) == 0:
    subprocess.run(
        [
            "kaggle", "datasets", "download",
            "-d", "sachinkumar413/diabetic-retinopathy-dataset",
            "-p", str(raw),
            "--unzip",
        ],
        check=True,
    )
if image_count(aptos) == 0:
    subprocess.run(
        [
            "kaggle", "datasets", "download",
            "-d", "sovitrath/diabetic-retinopathy-224x224-2019-data",
            "-p", str(aptos),
            "--unzip",
        ],
        check=True,
    )
print("Primary images", image_count(raw))
print("APTOS images", image_count(aptos))


## Point the split files at the Colab copies

The CSV files were written on your PC with local paths. This cell matches each row to the downloaded file by name. Train, validation, and test membership does not change.


In [ ]:
from pathlib import Path

import cv2
import pandas as pd

def index_images(*roots):
    buckets = {}
    for root in roots:
        for path in Path(root).rglob("*"):
            if path.suffix.lower() in {".png", ".jpg", ".jpeg"}:
                buckets.setdefault(path.name, []).append(path.resolve())
    return buckets

def choose(original: str, candidates):
    text = original.replace("\\", "/")
    if "aptos" in text or "external" in text or "colored_images" in text:
        ranked = [p for p in candidates if "colored_images" in str(p) or "aptos" in str(p)]
    else:
        ranked = [p for p in candidates if "/raw/" in str(p)]
    return ranked[0] if ranked else candidates[0]

buckets = index_images(ROOT / "data" / "raw", ROOT / "data" / "external")
processed = ROOT / "data" / "processed"
missing = []
for name in ("train.csv", "val.csv", "test.csv", "extra_train.csv"):
    csv_path = processed / name
    if not csv_path.exists():
        if name == "extra_train.csv":
            print("extra_train.csv is absent. Training uses the original train split only.")
            continue
        raise SystemExit(f"Missing {csv_path}")
    frame = pd.read_csv(csv_path)
    new_paths = []
    missed = 0
    for old in frame["image_path"]:
        candidates = buckets.get(Path(old).name, [])
        if not candidates:
            missed += 1
            missing.append(Path(old).name)
            new_paths.append(old)
        else:
            new_paths.append(str(choose(str(old), candidates)))
    frame["image_path"] = new_paths
    frame.to_csv(csv_path, index=False)
    print(f"{name}: {len(frame)} rows, missing {missed}")

if missing:
    raise SystemExit("Images missing after download. Examples: " + ", ".join(missing[:8]))

sample = pd.read_csv(processed / "train.csv").iloc[0]["image_path"]
image = cv2.imread(sample)
if image is None:
    raise SystemExit(f"Could not read {sample}")
print("Sample OK", sample, image.shape)


## Train

Same recipe as `scripts/train_accuracy.py`: extra APTOS rows on the training split only, up to 12 epochs per phase, early stop on validation accuracy, then one test score.


In [ ]:
import os
import runpy
import sys

import tensorflow as tf

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

runpy.run_path(str(ROOT / "scripts" / "train_accuracy.py"), run_name="__main__")


## Download the models

This saves `colab_models.zip` to your computer and, if Drive mounts, also to `MyDrive/dr_colab_models/`.

Send that zip back. It should contain `softmax_acc70.keras`, `coral_acc70.keras`, `acc70_report.json`, and `meta.json`. `best_model.keras` is included only when the new test accuracy beat 0.627.


In [ ]:
import json
import shutil
import zipfile
from pathlib import Path

from google.colab import files

bundle = Path("/content/colab_models")
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()
wanted = [
    ROOT / "models" / "softmax_acc70.keras",
    ROOT / "models" / "coral_acc70.keras",
    ROOT / "models" / "best_model.keras",
    ROOT / "models" / "meta.json",
    ROOT / "artifacts" / "experiments" / "acc70_report.json",
]
for path in wanted:
    if path.exists():
        shutil.copy(path, bundle / path.name)
        print(f"{path.name:24} {path.stat().st_size / 1e6:6.1f} MB")
    else:
        print(f"{path.name:24} not produced")

report = ROOT / "artifacts" / "experiments" / "acc70_report.json"
if report.exists():
    payload = json.loads(report.read_text())
    print("Softmax accuracy", round(payload["softmax"]["accuracy"], 3), "QWK", round(payload["softmax"]["qwk"], 3))
    print("CORAL   accuracy", round(payload["coral"]["accuracy"], 3), "QWK", round(payload["coral"]["qwk"], 3))
    print("Deployed", payload["deployed"], "winner", payload["winner"])

zip_out = Path("/content/colab_models.zip")
with zipfile.ZipFile(zip_out, "w", compression=zipfile.ZIP_STORED) as archive:
    for path in bundle.iterdir():
        archive.write(path, path.name)
print("Zip", zip_out, f"{zip_out.stat().st_size / 1e6:.1f} MB")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/dr_colab_models")
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_out, dest / "colab_models.zip")
    print("Drive copy", dest / "colab_models.zip")
except Exception as exc:
    print("Drive copy skipped:", exc)

files.download(str(zip_out))
